Convert nielson_conversion.json to nielson_data.csv

In [ ]:
# Make the Nielson JSON easy to read as a spreadsheet
from nielson_data import NielsonDataCSVConverter

# Define file paths
json_file = "nielson_ome.json"
csv_file = "nielson_data.csv"

# Create an instance of the converter
converter = NielsonDataCSVConverter(json_file, csv_file)

# Convert the JSON to CSV
converter.convert_to_csv()

Add priority assignments to each opioid in nielson_conversions.json

In [ ]:
# Import the class
from nielson_data import NielsonDataPriorityUpdater

# Define the JSON file path
json_file = "nielson_ome.json"

# Create an instance of the updater
updater = NielsonDataPriorityUpdater(json_file)

# Define the priority dictionary
priority_dict = {
    1: ['oral_hydromorphone', 'oral_morphine', 'oral_oxycodone','iv_hydromorphone','iv_morphine'],
    2: ['oral_codeine', 'oral_hydrocodone', 'oral_methadone', 'oral_oxymorphone',
        'oral_tapentadol','oral_tramadol','sublingual_buprenorphine','transdermal_fentanyl',
        'iv_fentanyl','sc_fentanyl']
}

# Update the priorities in the JSON file
updater.update_priorities(priority_dict)

Calculate the equivalent opioids from nielson_ome.json

In [ ]:
# Conversion Service
from opioid import Opioid
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS

conversion_service = ConversionService(NIELSON_CONVERSIONS)

iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone")
oral_morphine = Opioid(administration_method="oral", substance="morphine")

random_dose = conversion_service.generate_random_dose(iv_fentanyl)
print(f"Random dose for {iv_fentanyl.administration_method}_{iv_fentanyl.substance}: {random_dose:.2f} {iv_fentanyl.unit}")

equivalent_dose_oral_morphine = conversion_service.calculate_expected_dose(random_dose, iv_fentanyl, oral_morphine)
print(f"Equivalent dose for {oral_morphine.administration_method}_{oral_morphine.substance}: {equivalent_dose_oral_morphine:.2f} {oral_morphine.unit}")

Construct a prompt

In [ ]:
from prompt_service import PromptService
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS
from scenario_vignettes import SCENARIO_VIGNETTES
from scenario import Scenario, ScenarioPromptedConversion

# Initialize the PromptService
prompt_service = PromptService(ConversionService(NIELSON_CONVERSIONS))

# Select a scenario by ID (e.g., "v2")
scenario_dict = SCENARIO_VIGNETTES["v3"]
scenario = Scenario(scenario_dict)

# Create a ScenarioPromptedConversion object
scenario_conversion = ScenarioPromptedConversion(
    scenario,
    model="google/gemini-2.5-pro",
    include_nielson=True
)

# Generate the prompt using the PromptService instance
generated_prompt = scenario_conversion.generate_prompt(prompt_service)

# Print the generated prompt
print("Generated Prompt:")
print(generated_prompt)

# Print the sampled dose and calculated expected_to_dose
print(f"Expected Dose: {scenario_conversion.expected_to_dose:.2f} {scenario_conversion.to_opioid.unit}")

In [ ]:
from scenario_vignettes import SCENARIO_VIGNETTES
from scenario import Scenario, ScenarioPromptedConversion
from scenario_experiment import ScenarioExperiment
from prompt_service import PromptedOpioidConversion

PromptedOpioidConversion.initialize_client()

# Vignette IDs to test
vignette_ids = ["v2","v3","v4","v5","v6","v7","v8","v9"]

# Create ScenarioPromptedConversion objects for each vignette
scenarios = [
    ScenarioPromptedConversion(
        Scenario(SCENARIO_VIGNETTES[v_id]),
        model=None,  # Will be set by experiment
        include_nielson=False
    )
    for v_id in vignette_ids
]

# Prompt conditions: no Nielson, no examples
prompt_conditions = [
    {"include_nielson": False, "include_examples": False},
    {"include_nielson": True, "include_examples": False},
]

# All models to test
models = [
    'openai/gpt-4.1',
    'openai/gpt-4.1-mini',
    'openai/gpt-3.5-turbo',
    'anthropic/claude-3.5-sonnet:20241022',
    'mistralai/mistral-large-2407',
    'mistralai/mixtral-8x22b-instruct',
    'google/gemini-2.5-pro',
    'google/gemini-2.5-flash',
    'cohere/command-r-plus-08-2024',
    'ai21/jamba-large-1.7',
    'ai21/jamba-mini-1.7',
    'qwen/qwen-2.5-72b-instruct',
    'meta-llama/llama-3.3-70b-instruct',
    'deepseek/deepseek-chat'
]

# Specify output CSV filename
output_csv = "scenario_experiment_results.csv"

# Run the experiment (one repetition per scenario/model)
experiment = ScenarioExperiment(
    scenarios=scenarios,
    prompt_conditions=prompt_conditions,
    models=models,
    repetitions=7,
    filename=output_csv
)
experiment.run()

Ask the model for the equivalent opioid dose

In [2]:
# Run a single conversion with an instance of PromptedOpioidConversion
from conversion_service import ConversionService
from prompt_service import PromptService, PromptedOpioidConversion
from nielson_data import NIELSON_CONVERSIONS
from opioid import Opioid

# Initialize the OpenAI client once
PromptedOpioidConversion.initialize_client()

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/day")
oral_morphine = Opioid(administration_method="oral", substance="morphine", unit="mg/4 hours")
to_opioid = oral_morphine

# Create a PromptedOpioidConversion object
conversion = PromptedOpioidConversion(
    from_opioid=iv_fentanyl,
    to_opioid=to_opioid,
    model="deepseek/deepseek-chat",
    include_nielson=True,
    include_examples=True,
    ask_for_units=True,
    explain_units=True
)

# Generate the prompt
conversion.generate_prompt(prompt_service=PromptService(ConversionService(NIELSON_CONVERSIONS)))

# Query the model
observed_dose = conversion.ask_model()

# Print the results
print("Generated Prompt:")
print(conversion.prompt)
print(f"Observed Dose: {observed_dose:.2f} {to_opioid.unit}")
print(f"Expected Dose: {conversion.expected_to_dose:.2f} {to_opioid.unit}")
print(f"Ratio Observed/Expected: {conversion.ratio_observed_expected:.2f}")
print("LLM Response Text:")
print(conversion.response)  # <-- Add this line to show the raw LLM response

Generated Prompt:
Convert 186.86 mcg/day of fentanyl (iv) to mg/4 hours of morphine (oral).
How many mg/4 hours of morphine (oral) is equivalent to 186.86 mcg/day of fentanyl (iv)?

0.2 OME: 1 mcg/day of fentanyl (iv) is equivalent to 0.2 mg/day morphine (oral).
0.2 OME for fentanyl (iv) means 10 mg/day of morphine (oral) is equivalent to 50.00 mcg/day of fentanyl (iv).
And 120 mg/day of morphine (oral) is equivalent to 600.00 mcg/day of fentanyl (iv).

Notice that this is a request for the equivalent dose administered over 4 hours, not over 1 day. 
The equivalent dose over 4 hours should be 1/6 the total dose for 1 day.
Express the response in the format {'calculated dose': value, 'units': 'mg/4 hours'}. Do not include any additional text in your response, only the dictionary.
Observed Dose: 6.23 mg/4 hours
Expected Dose: 6.23 mg/4 hours
Ratio Observed/Expected: 1.00
LLM Response Text:
{'calculated dose': 6.23, 'units': 'mg/4 hours'}


Run an experiment converting opioids across prompting conditions and models

In [ ]:
# Run an experiment converting opioids across prompting conditions and models
from prompt_service import PromptedOpioidConversion
from opioid import OpioidPairGenerator
from experiment import Experiment

# Initialize all clients
PromptedOpioidConversion.initialize_client()

# Generate opioid pairs
opioid_list_ome = OpioidPairGenerator(lambda priority: priority <= 2 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=False, use_permutations=False)

opioid_pairs = OpioidPairGenerator(lambda priority: priority == 1 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=True, use_permutations=True)

# Define prompt conditions
prompt_conditions = [
    {"include_nielson": False, "include_examples": False},
    {"include_nielson": True, "include_examples": False}
]

# OpenRouter models
models = [
    "openai/gpt-4.1",
    "anthropic/claude-3.5-sonnet:20241022",
    "mistralai/mistral-large-2407",
    "mistralai/mixtral-8x22b-instruct",
    "google/gemini-2.5-pro",
    "cohere/command-r-plus-08-2024",
    "ai21/jamba-large-1.7",
    "ai21/jamba-mini-1.7",
    "qwen/qwen-2.5-72b-instruct",
    "meta-llama/llama-3.3-70b-instruct",
    "deepseek/deepseek-chat"
]

# Run OME experiment
experiment_ome = Experiment(opioid_list_ome, prompt_conditions, models, repetitions=18)
experiment_ome.run()

# Print the OME experiment ID
print(f"Experiment ID: {experiment_ome.experiment_id}")

# Run pairs experiment
experiment_pairs = Experiment(opioid_pairs, prompt_conditions, models, repetitions=18)
experiment_pairs.run()

# Print the pairs experiment ID
print(f"Experiment ID: {experiment_pairs.experiment_id}")

In [ ]:
# Run missing experiments for pairs without Nielson
from prompt_service import PromptedOpioidConversion
from opioid import OpioidPairGenerator
from experiment import Experiment

# Initialize client
PromptedOpioidConversion.initialize_client()

# Generate only direct pairs (not OME)
opioid_pairs = OpioidPairGenerator(lambda priority: priority == 3 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=False, use_permutations=False)

# Only models with missing data
models = [
        "openai/gpt-4.1",
        "openai/gpt-4.1-mini",
        "openai/gpt-3.5-turbo",
        "anthropic/claude-3.5-sonnet:20241022",
        "anthropic/claude-3-5-haiku-20241022",
        "mistralai/mistral-large-2407",
        "mistralai/mixtral-8x22b-instruct",
        "google/gemini-2.5-pro",
        "google/gemini-2.5-flash",
        "cohere/command-r-plus-08-2024",
        "ai21/jamba-large-1.7",
        "ai21/jamba-mini-1.7",
        "qwen/qwen-2.5-72b-instruct",
        "meta-llama/llama-3.3-70b-instruct",
        "deepseek/deepseek-chat"
    ]

# Only the condition without Nielson
prompt_conditions = [
    {"include_nielson": False, "include_examples": False},
    {"include_nielson": True, "include_examples": False}
]

# Run experiment for missing data
experiment = Experiment(opioid_pairs, prompt_conditions, models, repetitions=18)
experiment.run()

print(f"Experiment ID: {experiment.experiment_id}")

Organize all results by whether they include oral morphine

In [ ]:
from experiment_utils import add_ome_column
from prefix import harmonize_model_names
from vignette_model_result import VignetteModelResult, VignetteModelConsensus

add_ome_column(
    input_csv="scenario_experiment_results.csv",
    output_csv="scenario_experiment_results_by_ome.csv"
)

harmonize_model_names("scenario_experiment_results_by_ome.csv")

# Process all combinations in the scenario_experiment_results_by_ome.csv
results = VignetteModelResult.process_all_combinations(
    csv_file="scenario_experiment_results_by_ome.csv",
    output_file="scenario_model_analysis.csv",
    verbose=True
)

results = VignetteModelConsensus.process_all_combinations(
    csv_file="scenario_model_analysis.csv",
    output_file="vignette_model_consensus.csv",
    verbose=True
)

In [ ]:
from vignette_results_json import save_vignette_results_json

save_vignette_results_json()

In [ ]:
# Visualize a single opioid pair using the refactored OpioidPairVisualizer

from opioid_pair_visualizer import OpioidPairVisualizer
from bokeh.io import show

# Change to a different opioid pair
from_opioid = "oral_oxycodone"
to_opioid = "oral_morphine"
prompt_conditions = {"include_nielson": False, "include_examples": False}

# Create the visualizer instance
visualizer = OpioidPairVisualizer(
    experiment_results_csv="experiment_results_by_ome.csv",
    pair_model_csv="pair_model_analysis.csv",
    consensus_csv="opioid_pair_consensus.csv",
    from_opioid=from_opioid,
    to_opioid=to_opioid,
    prompt_conditions=prompt_conditions
)

# Show the visualization
layout = visualizer.create_plot(width=600, height=350)
show(layout)

In [ ]:
# Example usage for vignette visualization

from vignette_visualizer import VignetteVisualizer
from bokeh.io import show

# Set up your file paths and parameters
experiment_results_csv = "scenario_experiment_results_by_ome.csv"
vignette_model_csv = "scenario_model_analysis.csv"
consensus_csv = "vignette_model_consensus.csv"
vignette_id = "v2"  # Change to your vignette
prompt_conditions = {"include_nielson": False, "include_examples": False}

# Create the visualizer instance
visualizer = VignetteVisualizer(
    experiment_results_csv=experiment_results_csv,
    vignette_model_csv=vignette_model_csv,
    consensus_csv=consensus_csv,
    vignette_id=vignette_id,
    prompt_conditions=prompt_conditions,
)

# Show the visualization
layout = visualizer.create_plot(width=650, height=300)
show(layout)

Organize results by opioid pairs, find consensus across prompt conditions

In [ ]:
from opioid_pair_model_result import OpioidPairModelResult
from opioid_pair_consensus import OpioidPairConsensus
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS
from prefix import harmonize_model_names
from experiment_utils import add_ome_column

add_ome_column(
    input_csv="experiment_results.csv",
    output_csv="experiment_results_by_ome.csv"
)

harmonize_model_names("experiment_results_by_ome.csv")

# Initialize conversion service
conversion_service = ConversionService(NIELSON_CONVERSIONS)

# Process all combinations in the experiment_results_by_ome.csv
results = OpioidPairModelResult.process_all_combinations(
    csv_file="experiment_results_by_ome.csv",
    output_file="pair_model_analysis.csv",
    conversion_service=conversion_service,
    verbose=True
)

# Process all combinations in the pair_model_analysis.csv
results = OpioidPairConsensus.process_all_combinations(
    csv_file="pair_model_analysis.csv",
    output_file="opioid_pair_consensus.csv",
    conversion_service=conversion_service,
    verbose=True
)

# Update pair_model_analysis.csv with consensus results
updated_df = OpioidPairConsensus.update_pair_model_analysis(
    pair_model_csv="pair_model_analysis.csv",
    consensus_csv="opioid_pair_consensus.csv",
    output_csv="pair_model_analysis.csv",
    verbose=True
)

In [ ]:
# Make a CSV of model batch means for weight, tight, and broad ranges

from model_means import ModelBatchMeansBuilder

# Update combinations to use 'batch_name' instead of 'name'
combinations = [
    {"ome": None, "include_nielson": False, "batch_name": "all_n0"},
    {"ome": None, "include_nielson": True, "batch_name": "all_n1"},
    {"ome": True, "include_nielson": False, "batch_name": "ome_n0"},
    {"ome": True, "include_nielson": True, "batch_name": "ome_n1"},
    {"ome": False, "include_nielson": False, "batch_name": "pairs_n0"},
    {"ome": False, "include_nielson": True, "batch_name": "pairs_n1"},
]

builder = ModelBatchMeansBuilder("pair_model_analysis.csv")
builder.save_means_df(combinations, "model_batch_means.csv")

In [ ]:
# Use the current version of the heatmap visualizer to make a bar chart (no updates)

from batch import OpioidPairBatch
from opioid_pair_batch_heatmap_visualizer import OpioidPairBatchHeatmapVisualizer
from bokeh.plotting import show

# If you want to use the current method, you need to provide a batch object
# Example: create a batch for the desired batch_name
batch_name = "ome_n1"  # Change to your desired batch

# You may need to construct filter_specs to match your batch_name logic
filter_spec = {
    "include_nielson": True,  # Set according to batch_name
    "include_examples": False,
    "exclude_self_pairs": True,
    "ome": True  # Set according to batch_name
}
filter_specs = [filter_spec]

batch = OpioidPairBatch.from_filter_specs(
    experiment_results_csv="experiment_results_by_ome.csv",
    pair_model_csv="pair_model_analysis.csv",
    consensus_csv="opioid_pair_consensus.csv",
    filter_specs=filter_specs,
)

heatmap_vis = OpioidPairBatchHeatmapVisualizer(batch)
bar_chart = heatmap_vis._create_mean_accuracy_bar_chart(width=900, height=400)
show(bar_chart)

In [ ]:
from batch import OpioidPairBatch
from batch_visualizer import OpioidPairBatchVisualizer
from opioid_pair_batch_heatmap_visualizer import OpioidPairBatchHeatmapVisualizer
from visualization_config import convert_to_pdf_auto_size

combinations = [
   {"ome": None, "include_nielson": False, "name": "all_n0"},
   {"ome": None, "include_nielson": True, "name": "all_n1"},    
   {"ome": True, "include_nielson": False, "name": "ome_n0"},
   {"ome": True, "include_nielson": True, "name": "ome_n1"},  
   {"ome": False, "include_nielson": False, "name": "pairs_n0"},
   {"ome": False, "include_nielson": True, "name": "pairs_n1"},  
]

for combo in combinations:
    filter_spec = {
        "include_nielson": combo["include_nielson"],
        "include_examples": False,
        "exclude_self_pairs": True,
    }
    if combo["ome"] is not None:
        filter_spec["ome"] = combo["ome"]
    filter_specs = [filter_spec]
    
    try:
        batch = OpioidPairBatch.from_filter_specs(
            experiment_results_csv="experiment_results_by_ome.csv",
            variable_model_csv="pair_model_analysis.csv",
            consensus_csv="opioid_pair_consensus.csv",
            filter_specs=filter_specs,
        )

        vis = OpioidPairBatchVisualizer(
            batch,
            row_width=600,
            row_height=300,
        )

        batch_html = f"{combo['name']}_batch.html"

        vis.create_grid(output_html=batch_html)
        print(f"✓ Saved Bokeh HTML to {batch_html}")
        
        # Convert Bokeh to PDF
        batch_pdf = f"{combo['name']}_batch.pdf"
        await convert_to_pdf_auto_size(batch_html, batch_pdf, scale=2.0)
        print(f"✓ Saved Bokeh PDF to {batch_pdf}")

        heatmap_vis = OpioidPairBatchHeatmapVisualizer(
            batch,
            cell_size=35,
            width_padding=0,
            height_padding=0,
            cell_text_size="6pt",
            label_text_size="7pt"
        )

        # Save HTML
        output_html = f"{combo['name']}_heatmaps.html"
        heatmap_vis.create_heatmaps(output_html=output_html)
        print(f"✓ Saved HTML to {output_html}")
        
        # Convert to PDF
        output_pdf = f"{combo['name']}_heatmaps.pdf"
        await convert_to_pdf_auto_size(output_html, output_pdf, scale=3.0)
        print(f"✓ Saved PDF to {output_pdf}")

    except Exception as e:
        print(f"✗ Error creating {combo['name']}: {e}")

In [ ]:
from batch import VignetteBatch
from batch_visualizer import VignetteBatchVisualizer
from visualization_config import convert_to_pdf_auto_size

# Define filter specs for all vignettes and prompt conditions
filter_specs = [
    {"include_nielson": True, "include_examples": False},
    {"include_nielson": False, "include_examples": False},
]

# Create the batch (adjust file paths as needed)
batch = VignetteBatch.from_filter_specs(
    experiment_results_csv="scenario_experiment_results_by_ome.csv",
    variable_model_csv="scenario_model_analysis.csv",
    consensus_csv="vignette_model_consensus.csv",
    filter_specs=filter_specs,
)

# Visualize all combinations in one HTML
vis = VignetteBatchVisualizer(
    batch,
    row_width=900,
    row_height=300,
)
output_html = "all_vignette_batch.html"
vis.create_grid(output_html=output_html)

# Convert to PDF
output_pdf = "all_vignette_batch.pdf"
await convert_to_pdf_auto_size(output_html, output_pdf, scale=2.0)
print(f"✓ Saved batch PDF to {output_pdf}")

In [1]:
from experiment_utils import processing_pipeline

# For opioid pair experiments:
processing_pipeline(
    input_csv="experiment_results.csv",
    output_csv="experiment_results_by_ome.csv",
    variable_model_csv="pair_model_analysis.csv",
    consensus_csv="opioid_pair_consensus.csv",
    mode="pair"
)

Added 'ome' column to experiment_results_by_ome.csv
Harmonized model names written to experiment_results_by_ome.csv
Processed 10/1582 combinations...
Processed 20/1582 combinations...
Processed 30/1582 combinations...
Processed 40/1582 combinations...
Processed 50/1582 combinations...
Processed 60/1582 combinations...
Processed 70/1582 combinations...
Processed 80/1582 combinations...
Processed 90/1582 combinations...
Processed 100/1582 combinations...
Processed 110/1582 combinations...
Processed 120/1582 combinations...
Processed 130/1582 combinations...


/Users/tanyamarton/opioid-conversion-experiment/opioid_pair_model_result.py:127: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  slope, intercept, r_value, p_value, std_err = stats.linregress(dose_to, log_coefficients)


Processed 140/1582 combinations...
Processed 150/1582 combinations...
Processed 160/1582 combinations...
Processed 170/1582 combinations...
Processed 180/1582 combinations...
Processed 190/1582 combinations...
Processed 200/1582 combinations...
Processed 210/1582 combinations...
Processed 220/1582 combinations...
Processed 230/1582 combinations...
Processed 240/1582 combinations...
Processed 250/1582 combinations...
Processed 260/1582 combinations...
Processed 270/1582 combinations...
Processed 280/1582 combinations...
Processed 290/1582 combinations...
Processed 300/1582 combinations...
Processed 310/1582 combinations...
Processed 320/1582 combinations...
Processed 330/1582 combinations...
Processed 340/1582 combinations...
Processed 350/1582 combinations...
Processed 360/1582 combinations...
Processed 370/1582 combinations...
Processed 380/1582 combinations...
Processed 390/1582 combinations...
Processed 400/1582 combinations...
Processed 410/1582 combinations...
Processed 420/1582 c

In [1]:
from batch import OpioidPairBatch
from batch_visualizer import OpioidPairBatchVisualizer
from visualization_config import convert_to_pdf_auto_size

# Define the opioid pairs
pairs = [
    {'from_opioid': "iv_hydromorphone", 'from_opioid_unit': "mg/day", 'to_opioid': "oral_hydromorphone", 'to_opioid_unit': "mg/day"},
    {'from_opioid': "iv_hydromorphone", 'from_opioid_unit': "mg/day", 'to_opioid': "oral_morphine", 'to_opioid_unit': "mg/day"},
]

# Define filter specs for both include_nielson conditions
filter_specs = []
for include_nielson in [False, True]:
    for pair in pairs:
        filter_specs.append({
            "from_opioid": pair['from_opioid'],
            "from_opioid_unit": pair['from_opioid_unit'],
            "to_opioid": pair['to_opioid'],
            "to_opioid_unit": pair['to_opioid_unit'],
            "include_nielson": include_nielson,
            "include_examples": False,
            "exclude_self_pairs": True,
        })

# Create the batch
batch = OpioidPairBatch.from_filter_specs(
    experiment_results_csv="experiment_results_by_ome.csv",
    variable_model_csv="pair_model_analysis.csv",
    consensus_csv="opioid_pair_consensus.csv",
    filter_specs=filter_specs,
)

# Visualize all combinations in one HTML
vis = OpioidPairBatchVisualizer(
    batch,
    row_width=900,
    row_height=300,
)
output_html = "hydromorphone_pairs_batch.html"
vis.create_grid(output_html=output_html)

# Convert to PDF
output_pdf = "hydromorphone_pairs_batch.pdf"
await convert_to_pdf_auto_size(output_html, output_pdf, scale=2.0)
print(f"✓ Saved batch PDF to {output_pdf}")

✓ Generated PDF: 1745px x 1220px
✓ Saved batch PDF to hydromorphone_pairs_batch.pdf


In [ ]:
from opioid import Opioid
from prompt_service import PromptedOpioidConversion
from experiment import Experiment

# Define opioids with custom units
iv_hydromorphone = Opioid(administration_method="iv", substance="hydromorphone", unit="mg/day")
oral_hydromorphone_4hours = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/4 hours")
oral_morphine_4hours = Opioid(administration_method="oral", substance="morphine", unit="mg/4 hours")


# Create the pairs as tuples of Opioid objects
opioid_pairs = [
    (iv_hydromorphone, oral_hydromorphone_4hours),
    (iv_hydromorphone, oral_morphine_4hours),
]

# Only models with missing data
models = [
        "openai/gpt-4.1",
        "anthropic/claude-3.5-sonnet:20241022",
        "google/gemini-2.5-pro",
        "google/gemini-2.5-flash",
        "cohere/command-r-plus-08-2024",
        "deepseek/deepseek-chat"
    ]

# Prompt conditions
prompt_conditions = [
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": True},
]

# Initialize client
PromptedOpioidConversion.initialize_client()

# Run the experiment
experiment = Experiment(opioid_pairs, prompt_conditions, models, repetitions=25, filename="explain_units_experiment_results.csv")
experiment.run()

print(f"Experiment ID: {experiment.experiment_id}")


Experiment ID: 1537


In [7]:
from experiment_utils import processing_pipeline

# For opioid pair experiments:
processing_pipeline(
    input_csv="ask_for_units_experiment_results.csv",
    output_csv="ask_for_units_experiment_results_by_ome.csv",
    variable_model_csv="ask_for_units_pair_model_analysis.csv",
    consensus_csv="ask_for_units_opioid_pair_consensus.csv",
    mode="pair"
)

Added 'ome' column to ask_for_units_experiment_results_by_ome.csv
Harmonized model names written to ask_for_units_experiment_results_by_ome.csv
Processed 10/216 combinations...
Processed 20/216 combinations...
Processed 30/216 combinations...
Processed 40/216 combinations...
Processed 50/216 combinations...
Processed 60/216 combinations...
Processed 70/216 combinations...
Processed 80/216 combinations...
Processed 90/216 combinations...
Processed 100/216 combinations...
Processed 110/216 combinations...
Processed 120/216 combinations...
Processed 130/216 combinations...
Processed 140/216 combinations...
Processed 150/216 combinations...
Processed 160/216 combinations...
Processed 170/216 combinations...
Processed 180/216 combinations...
Processed 190/216 combinations...
Processed 200/216 combinations...
Processed 210/216 combinations...
Completed! Processed 216/216 combinations.
Results saved to: ask_for_units_pair_model_analysis.csv
Processed 16 consensus rows.
Skipped 0 pairs with no

In [6]:
from opioid_pair_visualizer import OpioidPairVisualizer
from bokeh.io import show

# File paths
experiment_results_csv = "custom_units_experiment_results.csv"
pair_model_csv = "custom_units_pair_model_analysis.csv"
consensus_csv = "custom_units_opioid_pair_consensus.csv"

# Specify opioid names and units
from_opioid = "iv_hydromorphone"
from_opioid_unit = "mg/day"
to_opioid = "oral_hydromorphone"
to_opioid_unit = "mg/4hr"
prompt_conditions = {
    "include_nielson": True,
    "include_examples": False
}

# Create and show the visualization
visualizer = OpioidPairVisualizer(
    experiment_results_csv=experiment_results_csv,
    pair_model_csv=pair_model_csv,
    consensus_csv=consensus_csv,
    from_opioid=from_opioid,
    from_opioid_unit=from_opioid_unit,
    to_opioid=to_opioid,
    to_opioid_unit=to_opioid_unit,
    prompt_conditions=prompt_conditions
)

layout = visualizer.create_plot(width=900, height=400)
show(layout)

In [9]:
from batch import OpioidPairBatch
from batch_visualizer import OpioidPairBatchVisualizer
from visualization_config import convert_to_pdf_auto_size
from bokeh.io import show

# File paths
experiment_results_csv = "ask_for_units_experiment_results.csv"
pair_model_csv = "ask_for_units_pair_model_analysis.csv"
consensus_csv = "ask_for_units_opioid_pair_consensus.csv"

# Extract all unique pairs and prompt conditions from the consensus file
import pandas as pd
df = pd.read_csv(consensus_csv)

filter_specs = []
for _, row in df.iterrows():
    filter_specs.append({
        "from_opioid": row["from_opioid"],
        "from_opioid_unit": row["from_opioid_unit"],
        "to_opioid": row["to_opioid"],
        "to_opioid_unit": row["to_opioid_unit"],
        "include_nielson": bool(row["include_nielson"]),
        "include_examples": bool(row["include_examples"]),
    })

# Create the batch and visualizer
batch = OpioidPairBatch.from_filter_specs(
    experiment_results_csv=experiment_results_csv,
    variable_model_csv=pair_model_csv,
    consensus_csv=consensus_csv,
    filter_specs=filter_specs,
)

vis = OpioidPairBatchVisualizer(
    batch,
    row_width=900,
    row_height=400,
)

# Save HTML
output_html = "ask_for_units_hydromorphone.html"
vis.create_grid(output_html=output_html)

# Convert HTML to PDF
output_pdf = "ask_for_units_hydromorphone.pdf"
await convert_to_pdf_auto_size(output_html, output_pdf, scale=2.0)


✓ Generated PDF: 1745px x 6420px


In [13]:
import pandas as pd
df = pd.read_csv("ask_for_units_experiment_results.csv")
df.columns

Index(['experiment_id', 'datetime', 'from_opioid', 'from_opioid_unit',
       'to_opioid', 'to_opioid_unit', 'from_dose', 'expected_to_dose', 'model',
       'prompt', 'include_nielson', 'include_examples', 'ask_for_units',
       'observed_to_dose', 'ratio_observed_expected', 'response'],
      dtype='object')

In [1]:
from opioid import Opioid
from opioid import OpioidPairGenerator
from prompt_service import PromptedOpioidConversion
from experiment import Experiment

# Generate OME opioids (priority <= 2)
from_opioids = OpioidPairGenerator(lambda priority: priority <= 4 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=False, use_permutations=False)

# Create oral morphine with custom unit
to_opioid = Opioid(administration_method="oral", substance="morphine", unit="mg/4 hours")

# Build pairs: each from_opioid paired with custom-unit oral morphine
opioid_pairs = [(from_opioid, to_opioid) for from_opioid in from_opioids]

# Models to use
models = [
        "openai/gpt-4.1",
        "openai/gpt-4.1-mini",
        "openai/gpt-3.5-turbo",
        "anthropic/claude-3.5-sonnet:20241022",
        "mistralai/mistral-large-2407",
        "mistralai/mixtral-8x22b-instruct",
        "google/gemini-2.5-pro",
        "google/gemini-2.5-flash",
        "cohere/command-r-plus-08-2024",
        "ai21/jamba-large-1.7",
        "ai21/jamba-mini-1.7",
        "qwen/qwen-2.5-72b-instruct",
        "meta-llama/llama-3.3-70b-instruct",
        "deepseek/deepseek-chat"
    ]

# Prompt conditions: always ask for units, always include Nielson
prompt_conditions = [
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": True},
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": False},
]

# Initialize client
PromptedOpioidConversion.initialize_client()

# Run the experiment and append results
experiment = Experiment(
    opioid_pairs,
    prompt_conditions,
    models,
    repetitions=25,
    filename="explain_units_experiment_results.csv"
)
experiment.run()

print(f"Experiment ID: {experiment.experiment_id}")

Error during conversion: Could not extract numeric value from response: 
Error during conversion: Could not extract numeric value from response: 
Error during conversion: Could not extract numeric value from response: 
Error during conversion: Could not extract numeric value from response: 
Error during conversion: Could not extract numeric value from response: 
Error during conversion: Could not extract numeric value from response: 
Error during conversion: Could not extract numeric value from response: `
        *   The user's hint confirms this: "The equivalent dose over
Experiment ID: 1540


In [1]:
from experiment_utils import processing_pipeline

# For opioid pair experiments:
processing_pipeline(
    input_csv="data/explain_units_experiment_results.csv",
    output_csv="data/explain_units_experiment_results_by_ome.csv",
    variable_model_csv="data/explain_units_pair_model_analysis.csv",
    consensus_csv="data/explain_units_opioid_pair_consensus.csv",
    mode="pair"
)

Added 'ome' column to data/explain_units_experiment_results_by_ome.csv
Harmonized model names written to data/explain_units_experiment_results_by_ome.csv
Processed 10/420 combinations...
Processed 20/420 combinations...
Processed 30/420 combinations...
Processed 40/420 combinations...
Processed 50/420 combinations...
Processed 60/420 combinations...
Processed 70/420 combinations...
Processed 80/420 combinations...
Processed 90/420 combinations...
Processed 100/420 combinations...
Processed 110/420 combinations...
Processed 120/420 combinations...
Processed 130/420 combinations...
Processed 140/420 combinations...
Processed 150/420 combinations...
Processed 160/420 combinations...
Processed 170/420 combinations...
Processed 180/420 combinations...
Processed 190/420 combinations...
Processed 200/420 combinations...
Processed 210/420 combinations...
Processed 220/420 combinations...
Processed 230/420 combinations...
Processed 240/420 combinations...
Processed 250/420 combinations...
Proce

In [5]:
import pandas as pd

raw_data_df = pd.read_csv("data/explain_units_experiment_results.csv")
print(f"Raw Data: {raw_data_df['explain_units'].value_counts()}")

pair_model_df = pd.read_csv("data/explain_units_pair_model_analysis.csv")
print(f"Pair Model Analysis: {pair_model_df['explain_units'].value_counts()}")

pair_consensus_df = pd.read_csv("data/explain_units_opioid_pair_consensus.csv")
print(f"Pair Consensus: {pair_consensus_df['explain_units'].value_counts()}")

Raw Data: explain_units
False    5248
True     5245
Name: count, dtype: int64
Pair Model Analysis: explain_units
False    420
True     420
Name: count, dtype: int64
Pair Consensus: explain_units
False    15
True     15
Name: count, dtype: int64


In [4]:
from opioid_model_comparison_bar_visualizer import ModelComparisonFigureBuilder

builder = ModelComparisonFigureBuilder()
await builder.save("model_comparison_bar.html", "model_comparison_bar.pdf")  # Bokeh bar charts
builder.save_scatter_figures_pdf("model_comparison_scatter.pdf") 

FileNotFoundError: [Errno 2] No such file or directory: 'model_batch_means.csv'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load your dataset
df = pd.read_csv("pair_model_analysis.csv")

# Extract raw weights
weights = df["weight"].dropna().astype(float)

# Sort for CDF
sorted_w = np.sort(weights)
cdf = np.arange(1, len(sorted_w) + 1) / len(sorted_w)

# Plot
plt.figure(figsize=(6, 4))
plt.plot(sorted_w, cdf, marker='.', linestyle='none')

plt.xlabel("Raw Weight")
plt.ylabel("CDF")
plt.title("CDF of Raw Model Weights Across Entire Dataset")
plt.grid(True, alpha=0.4)

plt.show()


In [ ]:
from opioid_model_comparison_bar_visualizer import OpioidModelComparisonBarVisualizer
from bokeh.layouts import row, column
from bokeh.io import output_file, save

combinations_list = [
    [
        {"ome": True, "include_nielson": False, "name": "ome_n0"},
        {"ome": True, "include_nielson": True, "name": "ome_n1"},
    ],
    [
        {"ome": False, "include_nielson": False, "name": "pairs_n0"},
        {"ome": False, "include_nielson": True, "name": "pairs_n1"},
    ],
    [
        {"ome": None, "include_nielson": False, "name": "all_n0"},
        {"ome": None, "include_nielson": True, "name": "all_n1"},
    ]
]

titles = [
    "OME Comparison",
    "Pairs Comparison",
    "All Comparison"
]

figures = []
for combos, title in zip(combinations_list, titles):
    visualizer = OpioidModelComparisonBarVisualizer.from_combinations(
        combos,
        experiment_results_csv="experiment_results_by_ome.csv",
        pair_model_csv="pair_model_analysis.csv",
        consensus_csv="opioid_pair_consensus.csv"
    )
    fig = visualizer.plot(title=title)
    figures.append(fig)

output_file("model_comparison_bar.html")
save(column(*figures))